# 09: A3 (abliteration) re-run with causal layer selection, Colab (PLAN.md §4, §6)
A3 v1 chose its layer by the largest Cohen's d separation of harmful vs benign dev prompts. On Tiny Aya that
statistic falls steadily after layer 4, so every model got the window floor (layer 7): a harmful-*content* direction
whose removal left refusal unchanged (A0 ~ A3 for all 8 models on harmful_test). A3 v2 follows Arditi et al. (2024,
App. C): for each layer in the 20-80% depth window, ablate its direction everywhere and measure the refusal rate
on 32 harmful_dev prompts; drop layers whose ablation moves benign_dev next-token distributions by KL > 0.1; the
lowest refusal wins. The weight edit itself (o_proj, down_proj, embeddings) is unchanged. Details:
`notes/limitations.md`, `src/aftermarket/attacks/abliterate.py`.

| step | runs | splits | flag |
|---|---|---|---|
| 1. archive v1 A3 results on the Hub (to `archive/A3_v1/`, nothing deleted) | all but the AMc_* selection record | all | none |
| 2. dev A3, read the per-layer table first | BASE, SDT | harmful_dev, benign_dev, mmlu_dev | none |
| 3. leaderboard A3 | BASE, SDT, GC, IL, RL, RL_noguard, ST, SP | harmful_test, xstest_test, mmlu_test | `--final` |
| 4. frozen AM family A3 (refuses unless AM is frozen + pinned) | AM, AM_rand0, AM_rand1, AM_all | same | `--final` |
| 5. judge (Llama Guard + WildGuard) and the A0 vs A3 table | everything above | | |

Test cells make no choice: the layer is picked per model on dev prompts only. One T4 (`--gpus 0`), same seed,
batch size and settings as before. About 3-4 GPU-h. Colab disconnects: just Run all again (step 1 finds nothing
left to archive; finished jobs are pulled from the Hub and skipped).

Requires: Colab secret `HF_TOKEN` (write; key icon, notebook access on), Runtime -> Change runtime type -> T4 GPU.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bennytimz/aftermarket/blob/main/notebooks/09_a3_colab.ipynb)


In [ ]:
# Session clock for the --max-hours guard
import time
SESSION_START = time.time()
MAX_HOURS = 10.0         # Colab costs no Kaggle quota. Re-run after a disconnect.
ARCHIVE_STALE = True     # step 1: move v1 A3 results aside on the Hub (only ones without select_by=bypass)


In [ ]:
# (a) Detect Colab vs Kaggle, then clone the repo (or update it when re-running in the same session). Non-interactive.
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
print("platform:", "Colab" if IN_COLAB else "Kaggle/other")
REPO_URL = "https://github.com/Bennytimz/aftermarket"
if os.path.basename(os.getcwd()) != "aftermarket":
    if not os.path.isdir("aftermarket/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("aftermarket")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
print("repo at", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# (b) Install only what Kaggle lacks (requirements-kaggle.txt), keeping the image's torch build fixed.
#     Colab: A1 trains a QLoRA adapter, so bitsandbytes / peft / accelerate are pinned as in requirements.txt.
#     torchao is removed: Kaggle's 0.10.0 breaks PEFT imports and we don't use it.
!pip uninstall -y -q torchao
!pip freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/keep-torch.txt
!pip install -q -r requirements-kaggle.txt -c /tmp/keep-torch.txt
if IN_COLAB:
    !pip install -q bitsandbytes==0.50.2 peft==0.21.2 accelerate==1.15.0 -c /tmp/keep-torch.txt
import importlib.metadata as md
for p in ["torch", "transformers", "peft", "accelerate", "huggingface_hub", "datasets", "bitsandbytes",
          "hqq", "curatorkit", "safetune", "circuitkit", "auditkit"]:
    try: print(f"{p:16s}", md.version(p))
    except md.PackageNotFoundError: print(f"{p:16s} MISSING")

In [ ]:
# (c) Log in to Hugging Face: Colab secrets (google.colab.userdata) on Colab, Kaggle secrets otherwise.
# Never prompts, so it also works in background "Save & Run All" runs.
from huggingface_hub import login, whoami
HF_USER = None
try:
    if IN_COLAB:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")  # key icon -> HF_TOKEN, notebook access on
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")  # Add-ons -> Secrets
    login(token=token, add_to_git_credential=False)
    HF_USER = whoami()["name"]
    print("HF user:", HF_USER)
except Exception as e:
    print(f"WARNING: no HF login ({type(e).__name__}: {e}). Gated models will fail; pushes/pulls are skipped.")

In [ ]:
# (d) GPU info
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")

In [ ]:
# (e) Pinned model revision
BASE_MODEL = "CohereLabs/tiny-aya-global"
BASE_REVISION = "af89d219b53ed9b13b8a4645f9c8028973510324"

In [ ]:
import os, json, sys
os.environ["AFTERMARKET_DEFENSE_DIR"] = "/kaggle/temp/aftermarket_defense" if os.path.isdir("/kaggle") else "/tmp/aftermarket_defense"
os.environ["AFTERMARKET_ADAPTERS"] = "/kaggle/temp/aftermarket_adapters" if os.path.isdir("/kaggle") else "/tmp/aftermarket_adapters"
os.environ["AFTERMARKET_TMP"] = "/kaggle/temp" if os.path.isdir("/kaggle") else "/tmp"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
for k in ("AFTERMARKET_DEFENSE_DIR", "AFTERMARKET_ADAPTERS"):
    os.makedirs(os.environ[k], exist_ok=True)
sys.path.insert(0, "src")
print("max hours:", MAX_HOURS)

## 1. Archive v1 A3 results (dry run first, then move; a no-op once done)

In [ ]:
!PYTHONPATH=src python -m aftermarket.attacks.a3_archive
if ARCHIVE_STALE:
    !PYTHONPATH=src python -m aftermarket.attacks.a3_archive --yes


## 2. Dev A3 (BASE, SDT)
Each model prints one line per candidate layer: refusal rate with that direction ablated vs none ablated, and KL on
benign prompts. If even the best layer leaves refusal near the no-ablation rate, A3 is genuinely weak on Tiny Aya
(a finding for the write-up), not a bug.

In [ ]:
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_cheap_dev.yaml --attacks A3 --gpus 0


In [ ]:
# Stats only (no completion text): the A3 layer choice per model, from the manifests
import glob, json
from aftermarket.judge import refusal_label
import pandas as pd

def a3_choices(pattern="results/gen/*__A3__harmful_*.json"):
    for p in sorted(glob.glob(pattern)):
        m = json.load(open(p)); a = m.get("attack_info") or {}
        df = pd.read_parquet(p[:-5] + ".parquet", columns=["response"])
        kw = (df.response.fillna("").map(refusal_label) == "refusal").mean()
        print(f"{m['job']['run_id']:12s} {m['job']['split']:13s} select_by={a.get('select_by')} layer={a.get('layer')} "
              f"| dev refusal none {a.get('refusal_no_ablation')} -> {a.get('refusal_at_layer')} "
              f"(KL {a.get('kl_at_layer')}{', KL filter relaxed' if a.get('kl_filter_relaxed') else ''}) "
              f"| keyword refusal on this split {kw:.2f}")
a3_choices()


## 3. Leaderboard A3 on test (`--final`; fixed attack on others' models, no choice made)

In [ ]:
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_cheap_test.yaml --final --attacks A3 --gpus 0


## 4. Frozen AM family under A3 (refuses unless AM is frozen and pinned on the Hub)

In [ ]:
!PYTHONPATH=src python -m aftermarket.defense.pipeline --config configs/defense.yaml --battery-only --part a3 --gpus 0 --max-hours {MAX_HOURS} --session-start {SESSION_START}


## 5. Judge (one worker on the T4) and A0 vs A3

In [ ]:
!PYTHONPATH=src python -m aftermarket.judge --config configs/judge.yaml --wildguard-on-harmful '*' --gpus 0


In [ ]:
# A0 vs A3 per model on harmful splits: Llama Guard ASR, WildGuard ASR, keyword refusal (stats only)
import os
rows = []
for p in sorted(glob.glob("results/judged/*__A3__harmful_*.parquet")):
    run, _, split = os.path.basename(p)[:-8].split("__")
    for att in ("A0", "A3"):
        q = f"results/judged/{run}__{att}__{split}.parquet"
        if not os.path.exists(q):
            continue
        d = pd.read_parquet(q)
        rows.append({"run": run, "split": split, "attack": att, "LG_ASR": d["unsafe"].mean() if "unsafe" in d else None,
                     "WG_ASR": pd.to_numeric(d.get("wg_harmful_response"), errors="coerce").mean() if "wg_harmful_response" in d else None,
                     "kw_refusal": (d.response.fillna("").map(refusal_label) == "refusal").mean()})
if rows:
    t = pd.DataFrame(rows).pivot_table(index=["run", "split"], columns="attack", values=["LG_ASR", "WG_ASR", "kw_refusal"])
    print(t.round(3).to_string())
else:
    print("nothing judged yet")
a3_choices()


## Status of the AM-family A3 cells

In [ ]:
!PYTHONPATH=src python -m aftermarket.defense.pipeline --config configs/defense.yaml --battery-only --part a3 --summary-only --max-hours {MAX_HOURS} --session-start {SESSION_START}
